In [1]:
import sys
sys.path.append('/workspace/python')

import duckdb
import polars as pl
import json
import utils_functions as utils

In [21]:
#Connect to DuckDB
con = duckdb.connect('/workspace/data/pipeline.duckdb')
print('Connected to DuckDB!')

Connected to DuckDB!


In [3]:
last_batch = con.execute('SELECT MAX(batch_id) FROM raw.api_data').fetchone()[0]

**MODIFY THE LINE OF CODE BELOW FOR DEBUGGING PURPOSES, OTHERWISE LEAVE IT ALONE SO IT ALWAYS LOADS THE LAST BATCH**

In [4]:
historical_batch_to_load = last_batch

In [5]:
# Adding the bronze table into a polars dataframe for further processing
df = con.execute(f"""
                 SELECT id, batch_id, page, raw_json 
                 FROM raw.api_data
                 WHERE batch_id = {historical_batch_to_load}
                 """).pl()

In [6]:
#Taking only the data column
df = df.with_columns([
    pl.col('raw_json')
      .map_elements(
          lambda x: [json.dumps(anime) for anime in json.loads(x)['data']], 
          return_dtype=pl.List(pl.String)
      )
      .alias('anime_list')
])

In [7]:
#Splitting each json with 25 animes into 25 rows with 1 anime each
df = df.explode('anime_list')

In [8]:
df = df.drop('raw_json').rename({
    'anime_list': 'anime_json',
    'id' : 'raw_id'
    })

In [9]:
records = [json.loads(x) for x in df['anime_json'].to_list()]
df_flattened = pl.json_normalize(records)

In [10]:
# Removing unnecessary columns
df_flattened = df_flattened[[
    'mal_id', 'title', 'title_english', 'type', 'source', 'episodes', 'status', 'duration', 'score', 'scored_by', 'rank', 'popularity',
    'members', 'favorites', 'year', 'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
    ]]

In [11]:
# Renaming columns
df_flattened = df_flattened.rename({
    'mal_id': 'anime_id',
    'episodes': 'no_of_episodes'
})

In [12]:
df_flattened = df_flattened.with_columns([
    pl.col('producers')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('producer_id'),
              pl.element().struct.field('name').alias('producer_name')
          )
      )
      .alias('producers'),
    
    pl.col('licensors')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('licensor_id'),
              pl.element().struct.field('name').alias('licensor_name')
          )
      )
      .alias('licensors'),

    pl.col('studios')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('studio_id'),
              pl.element().struct.field('name').alias('studio_name')
          )
      )
      .alias('studios'),

    pl.col('genres')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('genre_id'),
              pl.element().struct.field('name').alias('genre_name')
          )
      )
      .alias('genres'),

    pl.col('themes')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('theme_id'),
              pl.element().struct.field('name').alias('theme_name')
          )
      )
      .alias('themes'),

    pl.col('demographics')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('demographic_id'),
              pl.element().struct.field('name').alias('demographic_name')
          )
      )
      .alias('demographics')
])


In [13]:
concatted_df = pl.concat([df.drop('anime_json'), df_flattened], how='horizontal')

In [16]:
# Extracting bridge tables for genres, themes, demographics, studios, producers and licensors
links_anime_genres       = utils.extract_bridge_table(concatted_df, 'genres',       'genre_id',       'genre_name')
links_anime_themes       = utils.extract_bridge_table(concatted_df, 'themes',       'theme_id',       'theme_name')
links_anime_demographics = utils.extract_bridge_table(concatted_df, 'demographics', 'demographic_id', 'demographic_name')
links_anime_studios      = utils.extract_bridge_table(concatted_df, 'studios',      'studio_id',      'studio_name')
links_anime_producers    = utils.extract_bridge_table(concatted_df, 'producers',    'producer_id',    'producer_name')
links_anime_licensors    = utils.extract_bridge_table(concatted_df, 'licensors',    'licensor_id',    'licensor_name')

In [17]:
# Dropping the list[struct] columns as we already have all that information in the links tables
silver_anime = concatted_df.drop([
    'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
])

In [19]:
display(silver_anime.columns)

['raw_id',
 'batch_id',
 'page',
 'anime_id',
 'title',
 'title_english',
 'type',
 'source',
 'no_of_episodes',
 'status',
 'duration',
 'score',
 'scored_by',
 'rank',
 'popularity',
 'members',
 'favorites',
 'year']

In [15]:
import importlib
importlib.reload(utils)

<module 'utils_functions' from '/workspace/python/utils_functions.py'>

**The cells below overwrite the merge tables in DuckDB with our dataframes.** 

**You can run them one by one for debugging purposes, but the notebook will run all of them when called.**

In [23]:
utils.overwrite_table(con, silver_anime, 'curated.anime')

ConstraintException: Constraint Error: PRIMARY KEY or UNIQUE constraint violation: duplicate key "62568"

In [ ]:
utils.overwrite_table(con, links_anime_genres, 'curated.links_anime_genres')

In [ ]:
utils.overwrite_table(con, links_anime_themes, 'curated.links_anime_themes')

In [ ]:
utils.overwrite_table(con, links_anime_demographics, 'curated.links_anime_demographics')

In [ ]:
utils.overwrite_table(con, links_anime_studios, 'curated.links_anime_studios')

In [ ]:
utils.overwrite_table(con, links_anime_producers, 'curated.links_anime_producers')

In [ ]:
utils.overwrite_table(con, links_anime_licensors, 'curated.links_anime_licensors')

In [20]:
con.close()